# Sentiment Analysis via BERT Fine-Tuning

This notebook fine-tunes Google's `bert-base-uncased`, a bidirectional encoder-only transformer pretrained on masked language modeling, for binary sentiment classification on the Rotten Tomatoes movie review dataset.

Unlike decoder-only GPT-style models, BERT attends to both preceding and subsequent context tokens simultaneously, making it well-suited for tasks that require understanding an entire sequence rather than generating new text. The pretrained BERT backbone is extended with a lightweight classification head and fine-tuned end-to-end on labeled sentiment data.

The `transformers` and `datasets` packages from Hugging Face are required. Install them with `pip install transformers datasets` if not already present.

In [1]:
# pip install transformers

In [2]:
# pip install datasets

## 1. Tokenizer

The BERT tokenizer maps text to sequences of integer IDs from a 30,522-token vocabulary. Batches of different-length sequences are padded to the same length using `[PAD]` tokens, with an `attention_mask` marking which positions are real tokens (1) vs. padding (0). The attention mask is passed to BERT at every step so padding positions are ignored.

In [3]:
# Load BERT tokenizer and demonstrate encoding
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('google-bert/bert-base-uncased',  
                                          clean_up_tokenization_spaces=True)
print(tokenizer)
tokenized = tokenizer(["the cow", "jumped over the moon"], padding='longest', return_tensors="pt")
print(tokenized)

/home/users/gcc14/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BertTokenizer(name_or_path='google-bert/bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
{'input_ids': tensor([[  101,  1996, 11190,   102,     0,     0],
        [  101,  5598,  2058,  1996,  4231,   102]])

The tokenizer's `decode` method translates token IDs back to human-readable strings. Setting `skip_special_tokens=True` omits padding and other special tokens from the output.

In [4]:
# Decode token IDs back to text
for tokens in tokenized["input_ids"]:
    print(tokenizer.decode(tokens, skip_special_tokens=True))

the cow
jumped over the moon


## 2. Pretrained BERT Model

The pretrained `bert-base-uncased` model is loaded and inspected below. Key architectural properties:
- **Embedding dimension:** 768
- **Vocabulary size:** 30,522
- **Layers:** 12 transformer encoder blocks
- **Attention heads:** 12 per block
- **Parameters:** ~110M

In [5]:
# Load pretrained BERT model
import torch
from torch import nn
from transformers import BertModel
pretrained_model = BertModel.from_pretrained("google-bert/bert-base-uncased")
print(pretrained_model)

2024-11-13 18:51:00.542194: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

## 3. Sentiment Classification Architecture

`SentimentBert` wraps the pretrained BERT encoder with a single binary classification head:

```
BERT encoder → [CLS] hidden state (dim 768) → Linear(768→1) → Sigmoid → P(positive)
```

The `[CLS]` token's final hidden state encodes an aggregate representation of the entire input sequence — BERT's pretraining is designed to accumulate sequence-level meaning into this position, making it a natural input for classification.

In [6]:
import torch
from torch import nn
from transformers import BertModel

class SentimentBert(nn.Module):
    """
    BERT encoder with a binary sentiment classification head.
    Uses the [CLS] token's final hidden state as the sequence representation.
    """
    def __init__(self, pretrained_model):
        super().__init__()
        self.bert = pretrained_model
        self.classifier = nn.Linear(768, 1)

    def forward(self, tokenized):
        outputs = self.bert(
            input_ids=tokenized['input_ids'],
            attention_mask=tokenized['attention_mask']
        )
        cls_hidden = outputs.last_hidden_state[:, 0, :]  # [CLS] token representation
        return torch.sigmoid(self.classifier(cls_hidden))


In [7]:
# Verify the model runs on a small example before training
pretrained_model = BertModel.from_pretrained("google-bert/bert-base-uncased")
model = SentimentBert(pretrained_model)
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('google-bert/bert-base-uncased',
                                           clean_up_tokenization_spaces=True)
tokenized = tokenizer(["the cow", "jumped over the moon"], padding='longest', return_tensors="pt")
print("Output shape:", model(tokenized).shape)  # Should be [2, 1]
print(model(tokenized))

tensor([[0.5728],
        [0.5522]], grad_fn=<SigmoidBackward0>)


## 4. Dataset

The [Rotten Tomatoes](https://huggingface.co/datasets/rotten_tomatoes) dataset contains ~8,530 training and ~1,066 validation movie reviews, each labeled `1` (positive) or `0` (negative).

In [8]:
# pip install --upgrade --force-reinstall pyarrow

In [9]:
# Load train and validation splits from Rotten Tomatoes
from datasets import load_dataset
train_data = load_dataset("rotten_tomatoes", split="train")
val_data = load_dataset("rotten_tomatoes", split="validation")

print(f"Training examples: {len(train_data)}, Validation examples: {len(val_data)}")
for i in range(1, 3):
    print(train_data[i])
    print(train_data[-i])

Training examples: 8530, Validation examples: 1066
{'text': 'the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson\'s expanded vision of j . r . r . tolkien\'s middle-earth .', 'label': 1}
{'text': 'things really get weird , though not particularly scary : the movie is all portent and no content .', 'label': 0}
{'text': 'effective but too-tepid biopic', 'label': 1}
{'text': 'interminably bleak , to say nothing of boring .', 'label': 0}


Reviews vary in length, so padding the entire dataset to a fixed length wastes computation. A `collate_fn` passed to `DataLoader` handles per-batch dynamic padding: each batch is padded only to the length of its longest sequence.

In [10]:
from torch.utils.data import DataLoader
from transformers import BertTokenizer

def collate(batch):
    """
    Collation function for DataLoader: tokenizes a batch of reviews
    with dynamic padding to the longest sequence in the batch.
    """
    tokenizer = BertTokenizer.from_pretrained('google-bert/bert-base-uncased')
    texts = [element['text'] for element in batch]
    labels = [element['label'] for element in batch]

    tokens = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
    return {
        'input_ids': tokens['input_ids'],
        'attention_mask': tokens['attention_mask'],
        'labels': torch.tensor(labels)
    }

train_dataloader = DataLoader(train_data, batch_size=8, shuffle=True, collate_fn=collate)
val_dataloader = DataLoader(val_data, batch_size=8, shuffle=False, collate_fn=collate)

In [11]:
# Inspect one batch to verify DataLoader output
for batch in train_dataloader:
    print(batch)
    break

{'input_ids': tensor([[  101,  2066,  1996,  2190,  1997,  2643,  4232,  1005,  1055,  5691,
          1012,  1012,  1012,  2009,  2003, 17453, 16806, 12227,  1010, 22391,
          1010, 17727,  8625,  6494,  3468,  1012,   102,     0,     0,     0],
        [  101, 15835,  1996,  2095,  1005,  1055,  9033, 23697,  3367,  1998,
          2087,  4297, 11631,  7869,  3372,  3185,  1012,   102,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  2019,  4297, 11631,  7869,  3372, 18414, 19661,  1997,  1037,
          2143,  2008,  1005,  1055,  6524,  2004, 14036,  2004,  2009,  2071,
          2031,  2042,  1012,   102,     0,     0,     0,     0,     0,     0],
        [  101,  1996,  8562,  2003,  3140,  1998,  3082,  1011,  4375,  1010,
          1998,  5681,  3432, 16010,  1012,   102,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  2877,  1037,  331

## 5. Fine-Tuning

The full model (BERT backbone + classification head) is trained end-to-end on the sentiment data. Training loss is logged every 100 batches; validation loss and accuracy are evaluated after each epoch.

**Optimizer:** SGD, `lr=0.001`  
**Loss:** Binary Cross-Entropy  
**Epochs:** 3  
**Target:** ≥80% validation accuracy

In [12]:
# Fine-tune BERT for sentiment classification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = SentimentBert(pretrained_model)
model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr = 0.001)
loss_fn = nn.BCELoss()
epochs = 3

for epoch in range(epochs):
    for i, batch in enumerate(train_dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].float().to(device)

        x = {'input_ids': input_ids, 'attention_mask': attention_mask}

        #Forward propagation
        outputs = model(x)
        loss = loss_fn(outputs.squeeze(), labels)

        #Backward propagation
        loss.backward()

        #Gradient Step
        optimizer.step()
        optimizer.zero_grad()

        if (i + 1) % 100 == 0:
            print(f"Epoch: {epoch+1}, Batches: {i+1}, Loss: {loss.item()}")
    
    
    with torch.no_grad():
        val_loss = 0
        correct_predictions = 0
        for batch in val_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].float().to(device)

            x = {'input_ids': input_ids, 'attention_mask': attention_mask}
            outputs = model(x)

            loss = loss_fn(outputs.squeeze(), labels)
            val_loss += loss.item()

            predictions = (outputs.squeeze() >= 0.5).int() 
            correct_predictions += (predictions == labels).sum().item()

        val_loss = val_loss / len(val_dataloader)
        accuracy = correct_predictions / len(val_dataloader.dataset)

        print(f"Epoch {epoch+1}, Validation Loss: {val_loss:.4f}, Accuracy: {accuracy:.4f}")


Epoch: 1, Batches: 100, Loss: 0.6416524052619934
Epoch: 1, Batches: 200, Loss: 0.5954356789588928
Epoch: 1, Batches: 300, Loss: 0.5573590993881226
Epoch: 1, Batches: 400, Loss: 0.729947030544281
Epoch: 1, Batches: 500, Loss: 0.4259997606277466
Epoch: 1, Batches: 600, Loss: 0.1161687970161438
Epoch: 1, Batches: 700, Loss: 0.39869868755340576
Epoch: 1, Batches: 800, Loss: 0.34059634804725647
Epoch: 1, Batches: 900, Loss: 0.17300410568714142
Epoch: 1, Batches: 1000, Loss: 0.38199228048324585
Epoch 1, Validation Loss: 0.3334, Accuracy: 0.8565
Epoch: 2, Batches: 100, Loss: 0.31929656863212585
Epoch: 2, Batches: 200, Loss: 0.08028370887041092
Epoch: 2, Batches: 300, Loss: 0.11665815114974976
Epoch: 2, Batches: 400, Loss: 0.5529768466949463
Epoch: 2, Batches: 500, Loss: 0.43470340967178345
Epoch: 2, Batches: 600, Loss: 0.09353237599134445
Epoch: 2, Batches: 700, Loss: 0.3226476013660431
Epoch: 2, Batches: 800, Loss: 0.5311172604560852
Epoch: 2, Batches: 900, Loss: 0.32318270206451416
Epoch: 2

## 6. Error Analysis

Five misclassified validation examples are retrieved and examined to understand where the model fails and whether those failures are reasonable.

In [13]:
# Retrieve 5 misclassified validation examples

with torch.no_grad():
    for batch in val_dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].float().to(device)
        
        x = {'input_ids': input_ids, 'attention_mask': attention_mask}
        outputs = model(x)
        predictions = (outputs.squeeze() >= 0.5).int()

        for i in range(len(predictions)):
            if predictions[i] != labels[i]:
                print(f"Example {count}: {tokenizer.decode(input_ids[i].cpu(), skip_special_tokens=True)}")
                print(f"    Prediction: {predictions[i].item()}, True Label: {labels[i].item()}")
                count += 1

                # Stop after gathering 5 incorrect examples
                if count > 5:
                    break
        if count > 5:
            break


Example 1: made for teens and reviewed as such, this is recommended only for those under 20 years of age... and then only as a very mild rental.
    Prediction: 0, True Label: 1.0
Example 2: those moviegoers who would automatically bypass a hip - hop documentary should give " scratch " a second look.
    Prediction: 0, True Label: 1.0
Example 3: there's absolutely no reason why blue crush, a late - summer surfer girl entry, should be as entertaining as it is
    Prediction: 0, True Label: 1.0
Example 4: the events of the film are just so weird that i honestly never knew what the hell was coming next.
    Prediction: 0, True Label: 1.0
Example 5: mark pellington's latest pop thriller is as kooky and overeager as it is spooky and subtly in love with myth.
    Prediction: 0, True Label: 1.0


Most errors occur on reviews with genuinely ambiguous sentiment. Domain-specific film criticism language is the main challenge: phrases like "dark and disturbing" or "shockingly unpredictable" carry positive connotations within film reviews even though the individual words would typically signal negativity. The model — trained on surface-level token distributions — struggles to resolve this context-dependent inversion. These misclassifications reflect the inherent difficulty of the task rather than simple model failures.